<a href="https://colab.research.google.com/github/PranjanaNarayan/vehicle-anomaly-detection/blob/main/vehicle-anomaly-detection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install pandas numpy matplotlib seaborn scikit-learn tensorflow -q
print("All libraries ready!")

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
np.random.seed(42)

# Simulate 10000 sensor readings (like Tesla vehicle telemetry)
n = 10000
time = pd.date_range(start='2026-06-25', periods=n, freq='1min')

# Normal sensor readings
speed = np.random.normal(60, 15, n)
battery_voltage = np.random.normal(400, 5, n)
motor_temp = np.random.normal(75, 8, n)
battery_temp = np.random.normal(35, 4, n)
current_draw = np.random.normal(150, 20, n)

# Inject anomalies (real failures)
anomaly_idx = np.random.choice(n, 150, replace=False)
speed[anomaly_idx] = np.random.choice([0, 180, 200], len(anomaly_idx))
battery_voltage[anomaly_idx] = np.random.choice([200, 500, 600], len(anomaly_idx))
motor_temp[anomaly_idx] = np.random.choice([150, 200, 10], len(anomaly_idx))
battery_temp[anomaly_idx] = np.random.choice([80, 90, 100], len(anomaly_idx))
current_draw[anomaly_idx] = np.random.choice([400, 500, 0], len(anomaly_idx))

# Create labels
labels = np.zeros(n)
labels[anomaly_idx] = 1

# Create DataFrame
df = pd.DataFrame({
    'timestamp': time,
    'speed': speed,
    'battery_voltage': battery_voltage,
    'motor_temp': motor_temp,
    'battery_temp': battery_temp,
    'current_draw': current_draw,
    'is_anomaly': labels
})

print("Dataset created!")
print("Shape:", df.shape)
print("\nFirst 5 rows:")
print(df.head())
print(f"\nTotal anomalies injected: {int(labels.sum())}")
print(f"Anomaly percentage: {labels.mean()*100:.2f}%")

In [ ]:
fig, axes = plt.subplots(3, 2, figsize=(14, 10))
fig.suptitle('Tesla Vehicle Sensor Readings', fontsize=16)

sensors = ['speed', 'battery_voltage', 'motor_temp',
           'battery_temp', 'current_draw']
colors = ['#2196F3', '#4CAF50', '#FF5722', '#9C27B0', '#FF9800']

for i, (sensor, color) in enumerate(zip(sensors, colors)):
    ax = axes[i//2][i%2]
    ax.plot(df['timestamp'][:500], df[sensor][:500],
            color=color, alpha=0.7, linewidth=0.8)
    anomalies = df[df['is_anomaly']==1][:500]
    ax.scatter(anomalies['timestamp'][:500],
              anomalies[sensor][:500],
              color='red', s=50, zorder=5, label='Anomaly')
    ax.set_title(sensor.replace('_', ' ').title())
    ax.legend()

axes[2][1].axis('off')
plt.tight_layout()
plt.show()
print("Red dots = anomalies injected into sensor data!")

In [ ]:
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix

# Features for anomaly detection
features = ['speed', 'battery_voltage', 'motor_temp',
            'battery_temp', 'current_draw']
X = df[features]
y = df['is_anomaly']

# Scale the data
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print("Training Isolation Forest model...")
iso_forest = IsolationForest(
    contamination=0.015,
    n_estimators=100,
    random_state=42
)
iso_forest.fit(X_scaled)

# Predict anomalies (-1 = anomaly, 1 = normal)
predictions = iso_forest.predict(X_scaled)
df['predicted_anomaly'] = (predictions == -1).astype(int)

print("Model trained successfully!")
print(f"\nAnomalies detected: {df['predicted_anomaly'].sum()}")

In [ ]:
print("="*50)
print("Model Evaluation")
print("="*50)
print(classification_report(y, df['predicted_anomaly'],
      target_names=['Normal', 'Anomaly']))

cm = confusion_matrix(y, df['predicted_anomaly'])
plt.figure(figsize=(6,4))
sns.heatmap(cm, annot=True, fmt='d',
            xticklabels=['Normal', 'Anomaly'],
            yticklabels=['Normal', 'Anomaly'],
            cmap='Blues')
plt.title('Confusion Matrix — Sensor Anomaly Detection')
plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.show()

print(f"\nTrue Anomalies Caught: {cm[1][1]}")
print(f"Anomalies Missed: {cm[1][0]}")
print(f"False Alarms: {cm[0][1]}")

In [ ]:
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, LSTM, Dense, RepeatVector, TimeDistributed

print("Building LSTM Autoencoder...")

# Reshape for LSTM (samples, timesteps, features)
X_lstm = X_scaled.reshape(X_scaled.shape[0], 1, X_scaled.shape[1])

# Build autoencoder
inputs = Input(shape=(1, 5))
encoded = LSTM(32, activation='relu', return_sequences=False)(inputs)
decoded = RepeatVector(1)(encoded)
decoded = LSTM(32, activation='relu', return_sequences=True)(decoded)
outputs = TimeDistributed(Dense(5))(decoded)

autoencoder = Model(inputs, outputs)
autoencoder.compile(optimizer='adam', loss='mse')
autoencoder.summary()

print("Training LSTM Autoencoder...")
history = autoencoder.fit(
    X_lstm, X_lstm,
    epochs=10,
    batch_size=64,
    validation_split=0.1,
    verbose=1
)
print("Done!")

In [ ]:
# Calculate reconstruction error
X_pred = autoencoder.predict(X_lstm, verbose=0)
reconstruction_error = np.mean(np.abs(X_pred - X_lstm), axis=(1,2))

# Set threshold
threshold = np.percentile(reconstruction_error, 98)
df['lstm_anomaly'] = (reconstruction_error > threshold).astype(int)

print(f"Reconstruction Error Threshold: {threshold:.6f}")
print(f"Anomalies detected by LSTM: {df['lstm_anomaly'].sum()}")

# Plot reconstruction error
plt.figure(figsize=(12,4))
plt.plot(reconstruction_error[:500], color='blue',
         alpha=0.6, label='Reconstruction Error')
plt.axhline(y=threshold, color='red', linestyle='--',
            label=f'Threshold: {threshold:.4f}')
plt.title('LSTM Autoencoder — Reconstruction Error')
plt.xlabel('Time')
plt.ylabel('Error')
plt.legend()
plt.show()

print("\nLSTM Evaluation:")
print(classification_report(y, df['lstm_anomaly'],
      target_names=['Normal', 'Anomaly']))

In [ ]:
def check_vehicle_health(speed, battery_voltage,
                          motor_temp, battery_temp, current_draw):
    reading = np.array([[speed, battery_voltage,
                         motor_temp, battery_temp, current_draw]])
    reading_scaled = scaler.transform(reading)

    prediction = iso_forest.predict(reading_scaled)[0]
    score = iso_forest.score_samples(reading_scaled)[0]

    print("="*50)
    print("TESLA VEHICLE HEALTH CHECK")
    print("="*50)
    print(f"Speed:           {speed} km/h")
    print(f"Battery Voltage: {battery_voltage} V")
    print(f"Motor Temp:      {motor_temp} °C")
    print(f"Battery Temp:    {battery_temp} °C")
    print(f"Current Draw:    {current_draw} A")
    print("-"*50)

    if prediction == -1:
        print("⚠️  ANOMALY DETECTED!")
        print("🔧 Recommended Action: Schedule maintenance immediately")

        if motor_temp > 120:
            print("🌡️  Motor overheating — reduce speed")
        if battery_voltage < 300:
            print("🔋 Battery voltage critical — charge immediately")
        if battery_temp > 60:
            print("🔥 Battery temperature too high — stop vehicle")
    else:
        print("✅ Vehicle systems NORMAL")
        print("👍 No maintenance required")

    print(f"Anomaly Score: {score:.4f}")
    return prediction

# Test with normal reading
print("TEST 1 — Normal vehicle:")
check_vehicle_health(65, 400, 78, 36, 155)

print("\nTEST 2 — Anomalous vehicle:")
check_vehicle_health(185, 250, 160, 85, 450)

In [ ]:
import pickle
from google.colab import files

with open('anomaly_model.pkl', 'wb') as f:
    pickle.dump({
        'iso_forest': iso_forest,
        'scaler': scaler,
        'features': features
    }, f)

print("Model saved!")